# Comparaison des modèles récurrents sur HAR

Ce notebook lit les résultats existants de `exp/HAR` sans lancer d'entraînement : **axonal**, **synaptique**, **hybride**, et comparaison dédiée **axonal / hybride à offsets nuls**.

Les catégories viennent des `config.json` sauvegardés, jamais de la configuration actuelle. Les runs incomplets sont conservés pour leurs courbes, mais exclus des scores finaux. Les runs de deux époques ou moins sont exclus par défaut (heuristique pour les smoke tests).

**Interprétation :** le chargeur HAR actuel réutilise le même ensemble pour validation et test. Le score final n'est donc pas une mesure sur un test indépendant. Une seule seed ne permet pas de conclure à une supériorité statistique. Avec `max=0`, les offsets sont nuls, mais l'initialisation et les kernels peuvent différer de l'axonal ; les anciens runs peuvent aussi provenir du code qui remplaçait l'hybride zéro par un réseau axonal. Le nom du run ne suffit pas à identifier cette version du code.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'exp/HAR').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dépôt contenant exp/HAR.")
EXP = ROOT / 'exp/HAR'
INCLUDE_SHORT_RUNS = False
# None = tous les runs ; sinon une liste de noms de dossiers seed... à inclure.
RUN_NAMES = None
EXPORT = ROOT / 'figures' / 'har_comparison'
EXPORT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})
MODELS = {'SNN_axonal_recurrent_delays': 'Axonal',
          'SNN_synaptic_recurrent_delays': 'Synaptique',
          'SNN_recurrent_hybrid_delays': 'Hybride'}

def read_curve(path):
    if not path.exists():
        return pd.DataFrame(columns=['epoch', 'acc', 'loss'])
    raw = pd.read_csv(path, index_col=0)
    if raw.shape[0] != 2:
        raise ValueError(f"Format inattendu : {path}")
    return pd.DataFrame({'epoch': pd.to_numeric(raw.columns) + 1,
                         'acc': raw.iloc[0].to_numpy(dtype=float),
                         'loss': raw.iloc[1].to_numpy(dtype=float)}).sort_values('epoch')

runs = []
for model, family in MODELS.items():
    for path in sorted((EXP / model / 'perf').glob('*/config.json')):
        folder = path.parent
        if RUN_NAMES is not None and folder.name not in RUN_NAMES:
            continue
        cfg = json.loads(path.read_text())
        if not INCLUDE_SHORT_RUNS and cfg.get('epochs', 0) <= 2:
            continue
        maximum = cfg.get('hybrid_max_synaptic_delay') if family == 'Hybride' else None
        group = f'Hybride max={maximum}' if maximum is not None else ('Hybride max inconnu' if family == 'Hybride' else family)
        final_path = folder / 'final_test.json'
        final = json.loads(final_path.read_text()) if final_path.exists() else {}
        runs.append(dict(path=folder, config=cfg, final=final, family=family, group=group,
                         maximum=maximum, seed=cfg.get('seed'), name=folder.name,
                         train=read_curve(folder / 'train_res.csv'),
                         val=read_curve(folder / 'val_res.csv')))
if not runs:
    raise ValueError('Aucun run correspondant aux filtres.')
groups = list(dict.fromkeys(r['group'] for r in runs))
colors = dict(zip(groups, plt.get_cmap('tab10').colors))
def label(r):
    return f"{r['group']} | seed {r['seed']} | {r['name'].split('_', 1)[-1]}"
def savefig(name):
    plt.savefig(EXPORT / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()

summary = pd.DataFrame([{'groupe': r['group'], 'seed': r['seed'], 'run': r['name'],
    'époques observées': len(r['train']), 'terminé': bool(r['final']),
    'acc finale (%)': r['final'].get('acc', np.nan),
    'loss finale': r['final'].get('loss', np.nan),
    'meilleure époque (1-based)': r['final'].get('best_epoch', np.nan) + 1,
    'offset seed': r['config'].get('hybrid_delay_seed') if r['family'] == 'Hybride' else None
} for r in runs])
display(summary)
summary.to_csv(EXPORT / 'runs.csv', index=False)


## Scores finaux
Chaque barre représente un run terminé. Les répétitions d'une même seed restent distinctes ; aucune barre d'erreur ne leur attribue artificiellement le statut de seeds indépendantes.


In [ ]:
completed = [r for r in runs if r['final']]
if completed:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, key, title in zip(axes, ['acc', 'loss'], ['Accuracy finale (%)', 'Loss finale']):
        values = [r['final'][key] for r in completed]
        bars = ax.bar(range(len(completed)), values, color=[colors[r['group']] for r in completed])
        ax.set_xticks(range(len(completed)), [label(r) for r in completed], rotation=30, ha='right', fontsize=8)
        ax.bar_label(bars, fmt='%.3f', padding=3)
        ax.set_title(title)
        ax.set_ylim(0, 100 if key == 'acc' else max(values) * 1.2)
    fig.tight_layout()
    savefig('scores_finaux')
else:
    print('Aucun score final disponible.')


## Courbes d'apprentissage
Les époques sont affichées de 1 à N (les CSV utilisent 0 à N−1). Chaque courbe représente un run, sans lissage ni moyenne entre runs.


In [ ]:
def plot_learning(selected, filename):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for r in selected:
        for row, metric in enumerate(['acc', 'loss']):
            for col, split in enumerate(['train', 'val']):
                curve = r[split]
                axes[row, col].plot(curve['epoch'], curve[metric], label=label(r), color=colors[r['group']], alpha=.85)
    for row, metric in enumerate(['Accuracy (%)', 'Loss']):
        for col, split in enumerate(['Entraînement', 'Validation']):
            axes[row, col].set(title=f'{split} — {metric}', xlabel='Époque', ylabel=metric)
    axes[0, 0].legend(fontsize=7)
    fig.tight_layout()
    savefig(filename)
plot_learning(runs, 'courbes_tous_modeles')


## Axonal contre hybride à offsets nuls
La comparaison des scores ci-dessous associe les runs par seed et affiche toutes les paires disponibles. Un écart positif signifie une meilleure accuracy pour l'hybride zéro. Vérifier les configurations plus bas avant d'attribuer cet écart à l'architecture.


In [ ]:
zero_comparison = [r for r in runs if r['family'] == 'Axonal' or (r['family'] == 'Hybride' and r['maximum'] == 0)]
axonal = [r for r in zero_comparison if r['family'] == 'Axonal']
zero = [r for r in zero_comparison if r['family'] == 'Hybride']
if axonal and zero:
    plot_learning(zero_comparison, 'axonal_vs_hybride_zero')
    pairs = []
    fig, ax = plt.subplots(figsize=(11, 4))
    for a in axonal:
        for h in zero:
            if a['seed'] != h['seed']:
                continue
            joined = a['val'].merge(h['val'], on='epoch', suffixes=('_axonal', '_hybride'))
            ax.plot(joined['epoch'], joined['acc_hybride'] - joined['acc_axonal'],
                    label=f"{h['name']} − {a['name']}")
            if a['final'] and h['final']:
                pairs.append({'seed': a['seed'], 'axonal': a['name'], 'hybride zéro': h['name'],
                    'écart final (points)': h['final']['acc'] - a['final']['acc']})
    ax.axhline(0, color='black', linewidth=1)
    ax.set(xlabel='Époque', ylabel="Écart d'accuracy (points)", title='Hybride zéro − axonal : validation, mêmes seeds')
    if ax.lines and len(ax.lines) > 1:
        ax.legend(fontsize=7)
    fig.tight_layout()
    savefig('ecart_hybride_zero_axonal')
    display(pd.DataFrame(pairs))
else:
    print('Il faut au moins un run axonal et un run hybride max=0.')


## Différences de configuration
Comparaison des configurations archivées. Les chemins de sortie sont ignorés. « absent » signifie non enregistré : le notebook ne remplace pas une valeur manquante par une valeur actuelle. Les fichiers ne documentent pas nécessairement la version du code ou le kernel réellement utilisé.


In [ ]:
ignored = {'results_dir'}
keys = sorted(set().union(*(r['config'].keys() for r in runs)) - ignored)
differences = {}
for key in keys:
    values = [json.dumps(r['config'].get(key, '<absent>'), ensure_ascii=False, sort_keys=True) for r in runs]
    if len(set(values)) > 1:
        differences[key] = values
if differences:
    display(pd.DataFrame(differences, index=[label(r) for r in runs]).T)
else:
    print('Configurations enregistrées identiques hors chemins de sortie.')
print(f'Figures PNG et tableau des runs exportés dans : {EXPORT}')
